<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


# yoctoGPT — Finance Text Corpus (Token Mode)

This notebook trains `yoctoGPT` on a **finance text corpus** from the `finance/` folder.
Add one or more `.txt` files to `finance/` — more data improves quality.
Target setup: **Google Colab** (T4 default, GPU auto-detected).

## How to Use This Notebook

1. Add one or more finance `.txt` files to `finance/` (more data improves quality).
2. Clean the corpus and build tokenizer + dataset.
3. Train with GPU-adapted parameters.
4. Sample and score text quality.
5. Optionally resume from `best.pt` with lower LR.

**Hardware**: Designed for Google Colab (T4 default). GPU is auto-detected.
See `notebooks/colab_gpus.md` for the full GPU comparison table.

### Roadmap
- Setup and environment checks
- Multi-file finance tokenization (`finance/*.txt`)
- GPU-adapted training run
- Sampling + readability scoring
- Resume from best checkpoint

In [ ]:
#@title Mount Google Drive for checkpoint storage (repo stays local)
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

CKPT_DIR = Path('/content/drive/MyDrive/yocto/checkpoints/finance_token')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoints dir:', CKPT_DIR)

In [ ]:
#@title Setup: Install Dependencies and Clone Repository
!nvidia-smi || true
!pip -q install tokenizers tqdm textstat pyyaml

import os
import pathlib
import subprocess

repo_root = pathlib.Path('/content/yoctoGPT')
if repo_root.exists():
    print('Repo exists, pulling latest...')
    subprocess.run(['git', 'pull'], cwd=repo_root, check=False)
else:
    subprocess.run(['git', 'clone', 'https://github.com/yhilpisch/yoctoGPT.git', str(repo_root)], check=False)
os.chdir(repo_root)

if os.path.exists('requirements.txt'):
    !pip -q install -r requirements.txt || true

finance_dir = pathlib.Path('finance')
finance_dir.mkdir(exist_ok=True)
txts = sorted(finance_dir.glob('*.txt'))
if not txts:
    raise FileNotFoundError('No .txt files found in finance/. Add at least one corpus file.')

print(f'Found {len(txts)} finance text file(s):')
for fp in txts[:10]:
    print(' -', fp.name)
if len(txts) > 10:
    print(' - ...')

In [ ]:
#@title Detect GPU and load training profile
import subprocess, yaml

_out = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()

if "T4" in _out:
    GPU_KEY = "t4"
elif "L4" in _out:
    GPU_KEY = "l4"
elif "A100" in _out:
    GPU_KEY = "a100"
elif "RTX PRO 6000" in _out:
    GPU_KEY = "g4"
elif "H100" in _out:
    GPU_KEY = "h100"
else:
    GPU_KEY = "t4"

GPU_CFG = yaml.safe_load(open("notebooks/gpu_configs.yml"))[GPU_KEY]
AMP_DTYPE = GPU_CFG["amp_dtype"]
print(f"GPU: {_out} -> profile: {GPU_KEY}, amp_dtype: {AMP_DTYPE}")

### Tokenization

For technical finance text (formulas, symbols, and code-like fragments), token-level modeling is generally more effective than pure character-level modeling.

We first clean all `finance/*.txt` files to reduce back-matter/citation loops, then train one tokenizer on the cleaned corpus.

In [ ]:
#@title Clean Finance Corpus and Prepare Token-level Dataset
!python -m scripts.prepare_book_text \
--in_dir finance \
--out_dir finance_clean \
--glob '*.txt' \
--lowercase

!python -m scripts.prepare_tokenizer \
--all_txt_in_dir \
--text_dir finance_clean \
--out_dir data/token_finance \
--vocab_size 4000 \
--backend bpe \
--random_split \
--split_level chunk \
--split_seed 1337 \
--add_bos_eos

In [ ]:
#@title Check corpus size and load GPU profile
import numpy as np
from pathlib import Path

train_path = Path("data/token_finance/train.bin")
val_path = Path("data/token_finance/val.bin")
train_tokens = int(np.fromfile(train_path, dtype=np.int32).shape[0])
val_tokens = int(np.fromfile(val_path, dtype=np.int32).shape[0])
print(f"Train tokens: {train_tokens:,}, Val tokens: {val_tokens:,}")

P = GPU_CFG["token_finance"]
n_layer, n_head = P["n_layer"], P["n_head"]
n_embd = P["n_embd"]
block_size, batch_size = P["block_size"], P["batch_size"]
print(f"token_finance: n_layer={n_layer}, n_head={n_head}, n_embd={n_embd}, "
      f"block_size={block_size}, batch_size={batch_size}")


### Training

Small-corpus baseline (stable on single-book corpora): compact model + moderate
regularization + early stopping. Parameters are auto-selected from `gpu_configs.yml`.

As you add more books to `finance/`, you can scale model size and context.

In [ ]:
#@title Train (small-corpus baseline) on Colab GPU
!python -m yoctoGPT.train \
--mode token \
--data_dir data/token_finance \
--tokenizer_path data/token_finance/tokenizer.json \
--ckpt_dir {CKPT_DIR} \
--model_type gpt_fast \
--device cuda \
--n_layer {n_layer} --n_head {n_head} --n_embd {n_embd} \
--block_size {block_size} --batch_size {batch_size} \
--dropout 0.12 --weight_decay 0.08 \
--tie_weights --label_smoothing 0.05 \
--amp --amp_dtype {AMP_DTYPE} \
--auto_microbatch \
--eval_interval 200 --eval_iters 60 \
--cosine_lr --warmup_iters 100 \
--min_lr 1e-5 --lr 1.5e-4 \
--max_iters 2500 \
--early_stopping_patience 4 \
--early_stopping_min_delta 0.005

### Sampling and Resuming

Sample from `best.pt` and score text quality. Then optionally resume from `best.pt` with a lower LR.

In [ ]:
#@title Sample Continuation (anti-loop defaults)
output_token = !python -m yoctoGPT.sampler \
--mode token \
--ckpt {CKPT_DIR}/best.pt \
--tokenizer_path data/token_finance/tokenizer.json \
--prompt "in financial theory, the principle of no arbitrage implies" \
--max_new_tokens 80 \
--temperature 0.78 --top_k 12 --top_p 0.8

generated_text_token = '\n'.join(output_token)
print(generated_text_token)

### Text Quality Assessment

Same quality scorecard as other notebooks for direct comparison of output quality.

In [ ]:
#@title Score Generated Text Quality
from yoctoGPT.text_metrics import score_text, print_scorecard

# Build reference corpus from cleaned finance text
reference_corpus = '\n'.join(
    p.read_text(encoding='utf-8')
    for p in sorted(pathlib.Path('finance_clean').glob('*.txt'))
)

card = score_text(generated_text_token, reference_text=reference_corpus)
print_scorecard(card)


In [ ]:
#@title Resume training from best.pt (optional)
best = CKPT_DIR / "best.pt"
if not best.exists():
    print("No best.pt found; skipping resume cell.")
else:
    !python -m yoctoGPT.train \
      --mode token \
      --data_dir data/token_finance \
      --tokenizer_path data/token_finance/tokenizer.json \
      --ckpt_dir {CKPT_DIR} \
      --resume {best} \
      --model_type gpt_fast \
      --device cuda \
      --n_layer {n_layer} --n_head {n_head} --n_embd {n_embd} \
      --block_size {block_size} --batch_size {batch_size} \
      --dropout 0.12 \
      --tie_weights \
      --amp --amp_dtype {AMP_DTYPE} \
      --auto_microbatch \
      --cosine_lr --warmup_iters 80 \
      --lr 8e-5 --max_iters 800 \
      --eval_interval 200 --eval_iters 80 \
      --early_stopping_patience 3 \
      --early_stopping_min_delta 0.005

### Exercises

1. **Add Data**: Add another finance book to `finance/` and retrain. Compare best validation loss and sample coherence.
2. **Tokenizer Size**: Compare `--vocab_size 4000` vs `6000` on the same corpus.
3. **Model Scaling**: After adding more data, try `n_layer=6, n_embd=384, block_size=256` and compare output quality.

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
